## 导入依赖

In [1]:
import math
import numpy as np
import yaml
from kyle_robot_toolbox.transform import Transform
from kyle_robot_toolbox.handeye_calibration import icp_solver_svd
from kyle_robot_toolbox.robot_arm.arm4dof_uservo import Arm4DoFUServo

## 载入九点标定的配置文件

In [2]:
# 载入九点标定法的配置文件
handeye_9points_config = None
with open("config/handeye_calibration/handeye_9points.yaml", 'r', encoding='utf-8') as f:
    handeye_9points_config = yaml.load(f.read(), Loader=yaml.SafeLoader)

## 创建机械臂

In [3]:
# 创建机械臂
arm = Arm4DoFUServo(config_folder="./config", is_init_pose=False)

In [4]:
# 设置为阻尼模式
arm.set_damping(0)

## 获取采样点

将机械臂末端拖拽到9个点的位置，记录下机械臂末端的坐标值(x, y, z)

![](./image/p0.jpg)

In [5]:
p0 = arm.get_tool_pose()[:3]
p0

[73.06725503944817, 0.5023005662163278, 127.42110505272817]

![](./image/p1.jpg)

In [42]:
p1 = arm.get_tool_pose()[:3]
p1

j1:87.22339213893665,j2:23.97348131996793,j3:84.2743937681957,j4:41.13136033511588


[233, 11, -44.77]

![](./image/p2.jpg)

In [43]:
p2 = arm.get_tool_pose()[:3]
p2

j1:102.35111807566861,j2:24.278219089033982,j3:84.1663496737579,j4:39.29877496794441


[233, -51, -42.15]

![](./image/p3.jpg)

In [44]:
p3 = arm.get_tool_pose()[:3]
p3

j1:70.46799948749302,j2:16.659774862382974,j3:84.49048195707127,j4:55.58842267613527


[189, 67, -41.7]

In [45]:
p4 = arm.get_tool_pose()[:3]
p4

j1:90.2872353666292,j2:12.596604608169116,j3:84.38243786263348,j4:60.57712728676873


[193, -1, -34.5]

![](./image/p5.jpg)

In [46]:
p5 = arm.get_tool_pose()[:3]
p5

j1:104.45751029470723,j2:12.901342377235153,j3:84.49048195707127,j4:57.31919774513056


[196, -51, -31.06]

![](./image/p6.jpg)

In [47]:
p6 = arm.get_tool_pose()[:3]
p6

j1:66.44670525114655,j2:5.486056663294846,j3:84.38243786263348,j4:74.01608664602618


[147, 64, -28.81]

![](./image/p7.jpg)

In [48]:
p7 = arm.get_tool_pose()[:3]
p7

j1:86.36168623114813,j2:4.876581125162772,j3:84.05830557932012,j4:76.7649646967834


[153, 10, -28.95]

![](./image/p8.jpg)

In [49]:
p8 = arm.get_tool_pose()[:3]
p8

j1:106.85113781634205,j2:5.38447740693951,j3:84.38243786263348,j4:74.21970724237858


[153, -46, -28.73]

## 数据保存

In [50]:
ws_9p_at_arm = np.float32([p0, p1, p2, p3, p4, p5, p6, p7, p8])
ws_9p_at_arm

array([[233.  ,  71.  , -39.25],
       [233.  ,  11.  , -44.77],
       [233.  , -51.  , -42.15],
       [189.  ,  67.  , -41.7 ],
       [193.  ,  -1.  , -34.5 ],
       [196.  , -51.  , -31.06],
       [147.  ,  64.  , -28.81],
       [153.  ,  10.  , -28.95],
       [153.  , -46.  , -28.73]], dtype=float32)

将数据存放到csv文件中

In [51]:
np.savetxt('config/handeye_calibration/arm9points.txt', ws_9p_at_arm, delimiter=',', fmt='%.1f')

## Base标定

In [52]:
# 设置Numpy的打印选项
# 精确位数3，不启用科学计数法
np.set_printoptions(precision=3, suppress=True)

# 载入九点标定法的配置文件
handeye_9points_config = None
with open("config/handeye_calibration/handeye_9points.yaml", 'r', encoding='utf-8') as f:
	handeye_9points_config = yaml.load(f.read(), Loader=yaml.SafeLoader)


# 载入机械臂末端达到九个点
# 载入九点在2D图像中的坐标
arm_9points = np.loadtxt("config/handeye_calibration/arm9points.txt", delimiter=',')
# 主要注意的是，对于关节臂构型的机械臂
# 有的在Z轴上的误差很大，因此需要将
# arm_9points里面Z轴坐标替换为Z轴的均值。
z_mean = np.mean(arm_9points[:, 2])

# z轴误差纠正拟合
# - 转化为r跟z的形式
r = np.sqrt(arm_9points[:, 0]**2 + arm_9points[:, 1]**2)
z_error =  arm_9points[:, 2] - z_mean
z_error_polyfit = np.polyfit(r, z_error, 1)
np.savetxt("config/handeye_calibration/z_error_polyfit.txt", z_error_polyfit, fmt='%.4f', delimiter=",")
arm_9points[:, 2] = z_mean
print(f"9点在机械臂坐标系的坐标: \n{arm_9points}")

# 9点在工作台坐标系的坐标
# ws_9points_df = pd.read_excel("config/九点坐标-工作台坐标系.xlsx")
# ws_9points = np.float64(ws_9points_df.to_numpy())[:, 1:]
w = handeye_9points_config["board_width"]
h = handeye_9points_config["board_height"]
# P0点的坐标
x0 = 0.5 * h
y0 = 0.5 * w
ws_9points = np.float64([
	[x0, y0, 0], 	#P0
	[x0, 0, 0], 	#P1
	[x0, -y0, 0], 	#P2
 	[0, y0, 0], 	#P3
	[0, 0, 0], 		#P4
	[0, -y0, 0], 	#P5
	[-x0, y0, 0], 	#P6
	[-x0, 0, 0], 	#P7
	[-x0, -y0, 0], 	#P8
])
print(f"9点在工作台坐标系的坐标: \n{ws_9points}")

# 求解机械臂基坐标系到工作台坐标系的变换
R, t = icp_solver_svd(arm_9points, ws_9points)
T_arm2ws = np.eye(4)
T_arm2ws[:3, :3] = R
T_arm2ws[:3, 3] = t.reshape(-1)
print("T_arm2ws: \n{}".format(T_arm2ws))
np.savetxt("config/handeye_calibration/T_arm2ws.txt", T_arm2ws, fmt='%.3f', delimiter=",")

9点在机械臂坐标系的坐标: 
[[233.     71.    -35.556]
 [233.     11.    -35.556]
 [233.    -51.    -35.556]
 [189.     67.    -35.556]
 [193.     -1.    -35.556]
 [196.    -51.    -35.556]
 [147.     64.    -35.556]
 [153.     10.    -35.556]
 [153.    -46.    -35.556]]
9点在工作台坐标系的坐标: 
[[ 42.5  60.    0. ]
 [ 42.5   0.    0. ]
 [ 42.5 -60.    0. ]
 [  0.   60.    0. ]
 [  0.    0.    0. ]
 [  0.  -60.    0. ]
 [-42.5  60.    0. ]
 [-42.5   0.    0. ]
 [-42.5 -60.    0. ]]
T_arm2ws: 
[[  1.     -0.029   0.    192.222]
 [  0.029   1.      0.      8.222]
 [  0.      0.      1.    -35.556]
 [  0.      0.      0.      1.   ]]
